# ch15_implementation

*Computers in the Loop: Digital Control from Sampling to Networked Agents*: EECE 5610 lecture notes prepared and revised by Milad Siami.

Chapters 1–11 of the notes are based on the lecture notes of Prof. Mario Sznaier; all credits are in `ATTRIBUTION.md`.



Run the cells from top to bottom (in Colab: *Runtime > Run all*). Figures are also saved to a `figures` folder.


In [ ]:
#!/usr/bin/env python3
"""EECE 5610, Chapter 15 companion: implementing digital controllers.

Run it with
    python3 ch15_implementation.py
It prints the main numbers of the chapter and saves six figures (PNG) in a folder
ch15_figures/ next to this file:

  1. ch15_pm_vs_T.png        phase margin against T, with and without the anti-aliasing filter
  2. ch15_delay_steps.png    step responses with and without one sample of computation delay
  3. ch15_noise_gain.png     round-off noise in a first-order filter: formula against simulation
  4. ch15_coef_rounding.png  poles of a 4th-order filter after rounding: direct form vs cascade
  5. ch15_limit_cycles.png   deadband and limit cycle of a rounded first-order recursion
  6. ch15_windup.png         PI control with a saturating actuator: windup and anti-windup

Requires numpy, scipy and matplotlib. Change the parameters in the CAPITAL constants at the top
of each section to experiment.
"""

In [ ]:
import cmath
import math
import os

import numpy as np
from scipy import linalg, optimize

import matplotlib
import matplotlib.pyplot as plt

HERE = os.getcwd()
OUT = os.path.join(HERE, "ch15_figures")
os.makedirs(OUT, exist_ok=True)

In [ ]:
def save(fig, name):
    path = os.path.join(OUT, name)
    fig.tight_layout()
    fig.savefig(path, dpi=150)
    plt.close(fig)
    print(f"  saved {path}")

In [ ]:
# ---------------------------------------------------------------------------------------------
# Tools: zero-order-hold discretization and the phase margin of a sampled loop
# ---------------------------------------------------------------------------------------------
def zoh(A, B, T):
    """Exact ZOH discretization: Phi = e^{AT}, Gamma = int_0^T e^{As} ds B (one matrix exponential)."""
    n = A.shape[0]
    M = np.zeros((n + 1, n + 1))
    M[:n, :n] = A
    M[:n, n:] = B
    E = linalg.expm(M * T)
    return E[:n, :n], E[:n, n:]

In [ ]:
def plant_with_prefilter(wf=None):
    """State space of G_p(s) = 1/(s(s+1)), optionally followed by the Butterworth prefilter
    H_f(s) = wf^2 / (s^2 + sqrt(2) wf s + wf^2) on the measurement."""
    if wf is None:
        A = np.array([[0.0, 1.0], [0.0, -1.0]])
        B = np.array([[0.0], [1.0]])
        C = np.array([[1.0, 0.0]])
    else:
        A = np.array([[0, 1, 0, 0],
                      [0, -1, 0, 0],
                      [0, 0, 0, 1],
                      [wf ** 2, 0, -wf ** 2, -math.sqrt(2) * wf]], dtype=float)
        B = np.array([[0.0], [1.0], [0.0], [0.0]])
        C = np.array([[0.0, 0.0, 1.0, 0.0]])
    return A, B, C

In [ ]:
def loop_gain(T, w, K=1.0, wf=None, delay=0):
    """K z^-delay times the pulse transfer function of ZOH, plant (and prefilter), at z = e^{jwT}."""
    A, B, C = plant_with_prefilter(wf)
    Phi, Gam = zoh(A, B, T)
    z = cmath.exp(1j * w * T)
    G = (C @ np.linalg.solve(z * np.eye(A.shape[0]) - Phi, Gam))[0, 0]
    return K * G / z ** delay

In [ ]:
def phase_margin(T, K=1.0, wf=None, delay=0):
    """Find the crossover |L| = 1 below the Nyquist frequency and return (w_c, PM in degrees)."""
    wc = optimize.brentq(lambda w: abs(loop_gain(T, w, K, wf, delay)) - 1, 1e-3, math.pi / T * 0.999)
    pm = 180 + math.degrees(cmath.phase(loop_gain(T, wc, K, wf, delay)))
    if pm > 180:
        pm -= 360
    return wc, pm

In [ ]:
# ---------------------------------------------------------------------------------------------
# 1. Sampling period and anti-aliasing prefilter
# ---------------------------------------------------------------------------------------------
ATTENUATION_AT_NYQUIST = 0.1     # prefilter gain at w_s/2 (0.1 = -20 dB)

In [ ]:
def section_sampling_period():
    print("\n1. Sampling period and prefilter (plant 1/(s(s+1)), K = 1)")
    wc = math.sqrt((math.sqrt(5) - 1) / 2)          # analog crossover
    pm_analog = 90 - math.degrees(math.atan(wc))
    print(f"  analog design: w_c = {wc:.3f} rad/s, PM = {pm_analog:.1f} deg")
    # prefilter cutoff from |H_f(j w_N)| = 1/sqrt(1 + (w_N/w_f)^4) = ATTENUATION
    ratio = (1 / ATTENUATION_AT_NYQUIST ** 2 - 1) ** 0.25
    Ts = np.arange(0.02, 0.401, 0.02)
    pm_hold, pm_both = [], []
    for T in Ts:
        wf = (math.pi / T) / ratio
        pm_hold.append(phase_margin(T)[1])
        pm_both.append(phase_margin(T, wf=wf)[1])
    for T in (0.05, 0.1, 0.2):
        wf = (math.pi / T) / ratio
        print(f"  T = {T:4.2f} s: w_f = {wf:6.2f} rad/s, PM hold only = {phase_margin(T)[1]:.1f} deg,"
              f" hold + prefilter = {phase_margin(T, wf=wf)[1]:.1f} deg")
    fig, ax = plt.subplots(figsize=(6.4, 3.8))
    ax.plot(Ts, pm_hold, "o-", label="hold only (exact)")
    ax.plot(Ts, pm_both, "s-", label="hold and prefilter (exact)")
    ax.plot(Ts, pm_analog - np.degrees(wc * Ts / 2), "--", label="estimate: hold = delay T/2")
    k_pf = math.sqrt(2) * ratio / math.pi            # prefilter acts like a delay of k_pf * T
    ax.plot(Ts, pm_analog - np.degrees((0.5 + k_pf) * wc * Ts), "--",
            label=f"estimate: hold + prefilter = delay {0.5 + k_pf:.2f} T")
    ax.axhline(pm_analog, color="gray", lw=0.8, ls=":")
    ax.set_xlabel("T (s)")
    ax.set_ylabel("phase margin (deg)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    save(fig, "ch15_pm_vs_T.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 2. Computation delay in the running example (ZOH + 1/(s(s+1)), T = 1 s, D(z) = K)
# ---------------------------------------------------------------------------------------------
def step_response(K, delay, n=40):
    """c(k) of the unity-feedback loop with D(z) = K z^-delay, unit step reference."""
    a = math.exp(-1)
    b1, b0 = a, 1 - 2 * a                 # G(z) = (b1 z + b0) / ((z - 1)(z - a))
    c = np.zeros(n)
    m = np.zeros(n)                       # control applied to the plant
    for k in range(n):
        c[k] = ((1 + a) * c[k - 1] if k >= 1 else 0) - (a * c[k - 2] if k >= 2 else 0) \
            + (b1 * m[k - 1] if k >= 1 else 0) + (b0 * m[k - 2] if k >= 2 else 0)
        if delay:                         # m(k) computed at k is written at k + 1
            m[k] = K * (1 - c[k - 1]) if k >= 1 else 0.0
        else:
            m[k] = K * (1 - c[k])
    return c

In [ ]:
def largest_stable_gain(delay_fraction):
    """Largest K for D(z) = K when the output leaves delay_fraction * T after the sample (T = 1).
    State space with the delayed input: x(k+1) = Phi x + G0 m(k) + G1 m(k-1)."""
    A, B, C = plant_with_prefilter(None)
    T, tau = 1.0, delay_fraction
    Phi, _ = zoh(A, B, T)
    P1, G0 = zoh(A, B, T - tau)
    _, Gt = zoh(A, B, tau)
    G1 = P1 @ Gt
    lo, hi = 0.0, 5.0
    for _ in range(60):                   # bisection on the spectral radius
        K = (lo + hi) / 2
        Acl = np.zeros((3, 3))
        Acl[:2, :2] = Phi - K * G0 @ C
        Acl[:2, 2:] = G1
        Acl[2, :2] = -K * C
        if max(abs(np.linalg.eigvals(Acl))) < 1:
            lo = K
        else:
            hi = K
    return lo

In [ ]:
def section_delay():
    print("\n2. Computation delay (running example, T = 1 s)")
    for frac in (1e-9, 0.1, 0.2, 0.5, 1 - 1e-9):
        print(f"  output {frac:4.2f} T after the sample: stable for 0 < K < {largest_stable_gain(frac):.2f}")
    for K, d in [(0.5, 0), (0.5, 1), (0.3, 1)]:
        wc, pm = phase_margin(1.0, K=K, delay=d)
        c = step_response(K, d, 80)
        print(f"  K = {K}, delay = {d}: PM = {pm:.1f} deg, overshoot = {100 * (c.max() - 1):.1f} %")
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    for K, d, mk in [(0.5, 0, "o"), (0.5, 1, "s"), (0.3, 1, "^")]:
        ax.plot(step_response(K, d), mk + "-", ms=3, label=f"K = {K}, " + ("one sample of delay" if d else "no delay"))
    ax.axhline(1, color="gray", lw=0.8, ls=":")
    ax.set_xlabel("k")
    ax.set_ylabel("c(kT)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    save(fig, "ch15_delay_steps.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 3. Quantization noise through a first-order filter m(k) = Q[a m(k-1) + (1-a) e(k)]
# ---------------------------------------------------------------------------------------------
def Q(x):
    """Round to the nearest integer (units of q), ties away from zero."""
    return math.floor(abs(x) + 0.5) * (1 if x >= 0 else -1)

In [ ]:
def section_noise():
    print("\n3. Round-off noise in m(k) = Q[a m(k-1) + (1-a) e(k)]   (units of q)")
    rng = np.random.default_rng(1)
    poles = [0.5, 0.8, 0.9, 0.92, 0.95, 0.98]
    formula, simulated = [], []
    for a in poles:
        e = rng.uniform(-500, 500, 60000)          # a busy input, so the error looks random
        mq, mi = 0.0, 0.0
        err = []
        for ek in e:
            mq = Q(a * mq + (1 - a) * ek)          # what the processor computes
            mi = a * mi + (1 - a) * ek             # infinite precision
            err.append(mq - mi)
        formula.append(1 / (1 - a ** 2) / 12)      # noise gain times q^2/12
        simulated.append(np.var(err[2000:]))
        print(f"  a = {a:4.2f}: variance formula {formula[-1]:7.3f}, simulated {simulated[-1]:7.3f}")
    fig, ax = plt.subplots(figsize=(6.0, 3.6))
    ax.semilogy(poles, formula, "o-", label=r"$\frac{q^2}{12}\cdot\frac{1}{1-a^2}$")
    ax.semilogy(poles, simulated, "s", label="simulation with rounding")
    ax.set_xlabel("pole a")
    ax.set_ylabel(r"output noise variance (units of $q^2$)")
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3, which="both")
    save(fig, "ch15_noise_gain.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 4. Coefficient rounding: 4th-order filter with poles e^{-kT}, k = 1..4
# ---------------------------------------------------------------------------------------------
T_COEF = 0.05          # sampling period (s); try 0.1 and 0.02

In [ ]:
DECIMALS = 3           # decimals kept in each stored coefficient

In [ ]:
def section_coefficients():
    print(f"\n4. Coefficient rounding, poles e^(-kT), T = {T_COEF} s, {DECIMALS} decimals")
    p = np.exp(-T_COEF * np.arange(1, 5))
    a = np.poly(p)                                   # direct form: one 4th-order polynomial
    a_rounded = np.round(a, DECIMALS)
    direct = np.roots(a_rounded)
    cascade = np.round(p, DECIMALS)                  # first-order sections: coefficient = pole
    print(f"  exact poles         {np.round(p, 4)}")
    print(f"  A(1) exact {np.polyval(a, 1):.3e}, after rounding {np.polyval(a_rounded, 1):.3e}")
    print(f"  direct form poles   {np.round(direct, 4)}   max |z| = {max(abs(direct)):.4f}")
    print(f"  cascade poles       {cascade}   largest move = {max(abs(cascade - p)):.4f}")
    fig, ax = plt.subplots(figsize=(5.4, 4.2))
    th = np.linspace(-0.4, 0.4, 200)
    ax.plot(np.cos(th), np.sin(th), color="gray", lw=1, label="|z| = 1")
    ax.plot(p, 0 * p, "x", ms=9, mew=2, label="exact")
    ax.plot(direct.real, direct.imag, "x", ms=9, mew=2, label="direct form, rounded")
    ax.plot(cascade, 0 * cascade, "+", ms=11, mew=1.5, label="cascade, rounded")
    ax.set_aspect("equal")
    ax.set_xlim(0.6, 1.1)
    ax.set_ylim(-0.2, 0.2)
    ax.legend(fontsize=8, loc="center left")
    ax.grid(alpha=0.3)
    save(fig, "ch15_coef_rounding.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 5. Deadband and limit cycles: x(k+1) = Q[a x(k)]
# ---------------------------------------------------------------------------------------------
def section_limit_cycles():
    print("\n5. Rounded first-order recursion x(k+1) = Q[a x(k)], x(0) = 10")
    n = 15
    trunc = lambda x: math.floor(abs(x)) * (1 if x >= 0 else -1)   # magnitude truncation
    runs = {}
    for label, a, quant in [("a = 0.92, rounding", 0.92, Q), ("a = -0.92, rounding", -0.92, Q),
                            ("a = 0.92, truncation toward 0", 0.92, trunc)]:
        x = [10]
        for _ in range(n - 1):
            x.append(quant(a * x[-1]))
        runs[label] = x
        print(f"  {label:30s}: {x}")
    print(f"  deadband for |a| = 0.92: |x| <= 1/(2(1 - 0.92)) = {1 / (2 * 0.08):.2f}")
    fig, ax = plt.subplots(figsize=(6.4, 3.6))
    k = np.arange(n)
    ax.plot(k, 10 * 0.92 ** k, color="gray", label="no rounding")
    for (label, x), dx, mk in zip(runs.items(), (0, 0.2, -0.2), ("o", "s", "^")):
        ax.plot(k + dx, x, mk, label=label)
    ax.axhline(0, color="black", lw=0.6)
    ax.set_xlabel("k")
    ax.set_ylabel("x(k) / q")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    save(fig, "ch15_limit_cycles.png")

In [ ]:
# ---------------------------------------------------------------------------------------------
# 6. Saturation and windup: plant 1/(s+1), digital PI, |v| <= M_MAX
# ---------------------------------------------------------------------------------------------
T_PI, KP, KI, M_MAX, T_TRACK = 0.1, 4.0, 4.0, 1.2, 1.0

In [ ]:
def pi_loop(mode, t_end=8.0):
    """mode: 'none' (no limit), 'windup' (plain PI), 'back' (back-calculation), 'clamp'."""
    p = math.exp(-T_PI)                  # exact ZOH model of 1/(s+1)
    c, I = 0.0, 0.0
    C, Iout, V = [], [], []
    for _ in range(int(round(t_end / T_PI)) + 1):
        e = 1.0 - c                      # unit step reference
        m = KP * e + I                   # PI output, before the limit
        v = m if mode == "none" else max(-M_MAX, min(M_MAX, m))
        C.append(c)
        Iout.append(I)
        V.append(v)
        if mode == "back":               # back-calculation anti-windup
            I += KI * T_PI * e + (T_PI / T_TRACK) * (v - m)
        elif mode == "clamp":            # conditional integration
            if abs(m) <= M_MAX or m * e < 0:
                I += KI * T_PI * e
        else:
            I += KI * T_PI * e
        c = p * c + (1 - p) * v          # plant update over one period
    return np.array(C), np.array(Iout), np.array(V)

In [ ]:
def section_windup():
    print(f"\n6. PI with |v| <= {M_MAX}: plant 1/(s+1), T = {T_PI} s, Kp = Ki = {KP}")
    names = {"none": "no limit", "windup": "limit, plain PI", "back": "limit, back-calculation",
             "clamp": "limit, conditional integration"}
    colors = {"none": "gray", "windup": "tab:red", "back": "tab:blue", "clamp": "tab:green"}
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.8))
    for mode in ("none", "windup", "back", "clamp"):
        C, I, V = pi_loop(mode, t_end=12.0)
        t = T_PI * np.arange(len(C))
        settle = T_PI * (max(k for k in range(len(C)) if abs(C[k] - 1) > 0.02) + 1)
        print(f"  {names[mode]:32s}: peak c = {C.max():.3f}, largest I = {I.max():.2f},"
              f" 2% settling = {settle:.1f} s")
        keep = t <= 8.0
        ax1.plot(t[keep], C[keep], color=colors[mode], label=names[mode])
        if mode != "none":
            ax2.plot(t[keep], I[keep], color=colors[mode], label=names[mode])
    ax1.axhline(1, color="gray", lw=0.8, ls=":")
    ax2.axhline(M_MAX, color="gray", lw=0.8, ls="--")
    ax1.set_title("output c(kT)")
    ax2.set_title("integrator state I(k)")
    for ax in (ax1, ax2):
        ax.set_xlabel("t (s)")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)
    save(fig, "ch15_windup.png")

In [ ]:
if __name__ == "__main__":
    section_sampling_period()
    section_delay()
    section_noise()
    section_coefficients()
    section_limit_cycles()
    section_windup()
    print(f"\nFigures are in {OUT}")

## Explore




In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
r"""**Saturation and windup.** A digital PI controller ($K_p = K_i = 4$, $T = 0.1$ s) drives the
plant $1/(s+1)$ through an actuator limited to $|v| \le M$. With $M = 1.2$ the plain PI keeps
integrating while the actuator is stuck: $I$ climbs to 2.77 and $c$ overshoots to 1.176. Pick an
anti-windup scheme (orange) and compare. Lower $M$ toward 1 and watch the windup get worse; raise
it and the limit stops mattering. For back-calculation, the tracking time $T_t$ sets how fast
$I$ is pulled back.
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"
T, KP, KI = 0.1, 4.0, 4.0
T_RUN, T_SHOW = 12.0, 8.0


def _pi_loop(mode, M, Tt):
    """mode: 'none' (no limit), 'plain', 'back-calculation', 'conditional integration'."""
    p = np.exp(-T)                                     # exact ZOH model of 1/(s+1)
    c = I = 0.0
    C, Is, V = [], [], []
    for _ in range(int(round(T_RUN / T)) + 1):
        e = 1.0 - c                                    # unit step reference
        m = KP * e + I                                 # PI output before the limit
        v = m if mode == "none" else min(M, max(-M, m))
        C.append(c)
        Is.append(I)
        V.append(v)
        if mode == "back-calculation":
            I += KI * T * e + (T / Tt) * (v - m)
        elif mode == "conditional integration":
            if abs(m) <= M or m * e < 0:
                I += KI * T * e
        else:
            I += KI * T * e
        c = p * c + (1 - p) * v
    return np.array(C), np.array(Is), np.array(V)


def _settle(C):
    """2% settling time as text; the run lasts T_RUN seconds."""
    outside = np.nonzero(np.abs(C - 1) > 0.02)[0]
    if len(outside) and outside[-1] == len(C) - 1:
        return f"> {T_RUN:g} s"
    return f"{T * (outside[-1] + 1) if len(outside) else 0.0:.1f} s"


def explore(M=1.2, scheme="back-calculation", Tt=1.0):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.0))
    for mode, col, lab in [("none", GRAY, "no limit"), ("plain", BLUE, "limit, plain PI"),
                           (scheme, ORANGE, f"limit, {scheme}")]:
        C, I, V = _pi_loop(mode, M, Tt)
        t = T * np.arange(len(C))
        keep = t <= T_SHOW + 1e-9
        ls = "--" if mode == "none" else "-"
        ax1.plot(t[keep], C[keep], ls, color=col, lw=1.8, label=lab)
        ax2.plot(t[keep], I[keep], ls, color=col, lw=1.8, label=lab)
        stuck = int(np.sum(np.abs(V) >= M - 1e-12)) if mode != "none" else 0
        print(f"{lab:38s}: peak c = {C.max():.3f}, largest I = {I.max():.2f}, "
              f"2% settling = {_settle(C)}" + (f", samples at the limit = {stuck}" if mode != "none" else ""))
    ax1.axhline(1, color="k", lw=0.6)
    ax2.axhline(M, color="k", lw=0.8, ls=":", label=f"limit M = {M:g}")
    for ax in (ax1, ax2):
        ax.set_xlim(0, T_SHOW)
        ax.set_xlabel("t (s)")
        ax.legend(fontsize=10, loc="lower right")
    ax1.set_ylabel("c(kT)")
    ax2.set_ylabel("integrator state I(k)")
    extra = f", $T_t$ = {Tt:g} s" if scheme == "back-calculation" else ""
    ax1.set_title(f"output, |v| $\\leq$ {M:g}{extra}", fontsize=11)
    ax2.set_title("integrator state", fontsize=11)
    fig.tight_layout()
    plt.show()


try:
    from ipywidgets import interact, FloatSlider, Dropdown
    interact(explore,
             M=FloatSlider(value=1.2, min=1.05, max=4.0, step=0.05, description="limit M",
                           continuous_update=False),
             scheme=Dropdown(options=["back-calculation", "conditional integration"],
                             value="back-calculation", description="anti-windup"),
             Tt=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description="T_t (s)",
                            continuous_update=False))
except ImportError:
    explore()

## Hands-on

**Hands-on, Chapter 15.** These cells go with the Hands-on exercises at the end of
Chapter 15. Do each exercise on paper first and write down your prediction; then use the
matching section below to check it. Each section has sliders: change one thing at a time.


In [ ]:
# In the browser (JupyterLite) the sliders need one extra package.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install("ipywidgets")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"font.size": 11})
ORANGE, BLUE, TEAL, GRAY = "#E67E22", "#1D5FA8", "#1F6E78", "#6B7785"

In [ ]:
# ---------------------------------------------------------------- H1
def h1_averaging(N=16, signal="moving"):
    """H1: 12-bit A/D over 0-20 A; average N readings per period T = 1 ms."""
    q, T, periods = 20 / 4096, 1e-3, 400
    Tf = T / N
    t = (np.arange(periods * N) + 1) * Tf
    if signal == "moving":
        x = 10 + 2 * np.sin(2 * np.pi * 50 * t) + 0.7 * np.sin(2 * np.pi * 173 * t)
    else:
        x = np.full_like(t, 7.502)
    read = np.round(x / q) * q
    err_one = read - x
    err_avg = (read.reshape(periods, N).mean(axis=1) - x.reshape(periods, N).mean(axis=1))
    r1, rN = np.sqrt(np.mean(err_one ** 2)), np.sqrt(np.mean(err_avg ** 2))
    delay = (N - 1) / 2 * Tf
    fig, ax = plt.subplots(figsize=(10, 3.6))
    k = np.arange(60)
    ax.plot(k, 1e3 * err_one[(k + 1) * N - 1], "o", color=GRAY, ms=4, label="one reading")
    ax.plot(k, 1e3 * err_avg[k], "s-", color=ORANGE, ms=4, lw=1, label=f"average of {N}")
    ax.axhspan(-1e3 * q / 2, 1e3 * q / 2, color=BLUE, alpha=0.08)
    ax.set_xlabel("k")
    ax.set_ylabel("quantization error (mA)")
    ax.legend(loc="upper right", fontsize=9)
    bits = np.log2(r1 / rN) if rN > 0 else np.inf
    ax.set_title(f"{signal}: rms {1e3 * r1:.3f} -> {1e3 * rN:.3f} mA ({bits:.2f} bits), "
                 f"delay {1e3 * delay:.3f} ms = {np.degrees(200 * delay):.1f} deg at 200 rad/s",
                 fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H2
def h2_roundoff(extra_bits=0, a=0.96):
    """H2: m(k) = Q[a m(k-1) + (1-a) e(k)], A/D 12 bits on 0-10 V, internal step q / 2^extra_bits."""
    q = 10 / 4096
    qi = q / 2 ** extra_bits
    n = 40000
    rng = np.random.default_rng(1)          # load cell with vibration: the signal crosses many levels
    e_true = 5 + 2 * np.sin(2 * np.pi * np.arange(n) / 997.0) + 0.2 * rng.standard_normal(n)
    e_ad = np.round(e_true / q) * q
    m_ideal = np.zeros(n)        # no A/D, no rounding
    m_ad = np.zeros(n)           # A/D only
    m_both = np.zeros(n)         # A/D and internal rounding
    for k in range(1, n):
        m_ideal[k] = a * m_ideal[k - 1] + (1 - a) * e_true[k]
        m_ad[k] = a * m_ad[k - 1] + (1 - a) * e_ad[k]
        m_both[k] = np.round((a * m_both[k - 1] + (1 - a) * e_ad[k]) / qi) * qi
    s = slice(2000, n)
    r_ad = np.sqrt(np.mean((m_ad[s] - m_ideal[s]) ** 2))
    r_ro = np.sqrt(np.mean((m_both[s] - m_ad[s]) ** 2))
    fig, ax = plt.subplots(figsize=(10, 3.6))
    kk = np.arange(3000, 3300)
    ax.plot(kk, 1e3 * (m_ad[kk] - m_ideal[kk]), color=BLUE, lw=1.2, label="A/D noise at m")
    ax.plot(kk, 1e3 * (m_both[kk] - m_ad[kk]), color=ORANGE, lw=1.2, label="round-off noise at m")
    ax.set_xlabel("k")
    ax.set_ylabel("mV")
    ax.legend(loc="upper right", fontsize=9)
    ax.set_title(f"a = {a}, {extra_bits} extra bits: rms A/D {1e3 * r_ad:.3f} mV, "
                 f"round-off {1e3 * r_ro:.3f} mV", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H3
def h3_fractional_delay(delta=0.25, c=1.0):
    """H3: theta' = u, u(k) = K (r - theta(kT)) written delta*T after the sample; c = K T."""
    n, r = 40, 1.0
    th = np.zeros(n + 1)
    u_prev = 0.0
    for k in range(n):
        u = c * (r - th[k])                     # u in units of 1/T
        th[k + 1] = th[k] + (1 - delta) * u + delta * u_prev
        u_prev = u
    roots = np.roots([1, c * (1 - delta) - 1, c * delta])
    rho = max(abs(roots))
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [2, 1]})
    ax1.plot(np.arange(n + 1), th, "o-", color=ORANGE, ms=4, lw=1)
    ax1.axhline(r, color="k", lw=0.6)
    ax1.set_ylim(-1, 3)
    ax1.set_xlabel("k")
    ax1.set_ylabel("theta(k)")
    ax1.set_title(f"delta = {delta:.2f}, c = {c:.2f}: largest |root| = {rho:.3f} "
                  f"({'decays' if rho < 1 else 'does not decay'})", fontsize=11)
    w = np.linspace(0, 2 * np.pi, 300)
    ax2.plot(np.cos(w), np.sin(w), color=GRAY, lw=1)
    ax2.plot(roots.real, roots.imag, "x", color=ORANGE, ms=12, mew=2.5)
    ax2.set_aspect("equal")
    ax2.set_xlim(-1.6, 1.6)
    ax2.set_ylim(-1.3, 1.3)
    ax2.set_title("roots", fontsize=11)
    plt.show()

In [ ]:
# ---------------------------------------------------------------- H4
def h4_phase_budget(T_ms=1.2, schedule="(c) compute early"):
    """H4: wc = 60 rad/s, PM 55 deg; hold + Butterworth prefilter (0.1 at wN) + computation delay."""
    wc, pm0, tau_c = 60.0, 55.0, 0.4e-3

    def lags(T, sched):
        wf = (np.pi / T) / 99 ** 0.25
        x = wc / wf
        hold = wc * T / 2
        pre = np.arctan2(np.sqrt(2) * x, 1 - x ** 2)
        dly = wc * (T if sched.startswith("(b)") else 0.05e-3)
        return np.degrees([hold, pre, dly])

    T = T_ms * 1e-3
    h, p, d = lags(T, schedule)
    pm = pm0 - h - p - d
    load = tau_c / T
    Ts = np.linspace(0.3e-3, 2.0e-3, 200)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 3.8), gridspec_kw={"width_ratios": [1, 2]})
    ax1.bar(["hold", "prefilter", "delay"], [h, p, d], color=[BLUE, TEAL, ORANGE])
    ax1.set_ylabel("lag at wc (deg)")
    for sched, col in [("(b) next tick", GRAY), ("(c) compute early", BLUE)]:
        ax2.plot(1e3 * Ts, [pm0 - sum(lags(Tk, sched)) for Tk in Ts], color=col, lw=1.6, label=sched)
    ax2.plot(T_ms, pm, "o", color=ORANGE, ms=9)
    ax2.axhline(45, color="k", ls="--", lw=0.8)
    ax2.axvline(2 * tau_c * 1e3, color="k", ls=":", lw=0.8)
    ax2.set_xlabel("T (ms)")
    ax2.set_ylabel("PM estimate (deg)")
    ax2.legend(loc="upper right", fontsize=9)
    fig.suptitle(f"T = {T_ms:.1f} ms, {schedule}: PM {pm:.1f} deg "
                 f"({'yes' if pm >= 45 else 'no'}), load {100 * load:.0f}% "
                 f"({'yes' if load <= 0.5 + 1e-12 else 'no'})", fontsize=11)
    plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    interact(h1_averaging, N=IntSlider(value=16, min=1, max=256, description="N",
                                       continuous_update=False),
             signal=Dropdown(options=["moving", "steady"], value="moving", description="signal"))
    interact(h2_roundoff, extra_bits=IntSlider(value=0, min=0, max=8, description="extra bits",
                                               continuous_update=False),
             a=Dropdown(options=[0.8, 0.92, 0.96, 0.99], value=0.96, description="pole a"))
    interact(h3_fractional_delay,
             delta=FloatSlider(value=0.25, min=0.0, max=1.0, step=0.05, description="delta",
                               continuous_update=False),
             c=FloatSlider(value=1.0, min=0.1, max=5.0, step=0.05, description="c = KT",
                           continuous_update=False))
    interact(h4_phase_budget,
             T_ms=FloatSlider(value=1.2, min=0.3, max=2.0, step=0.1, description="T (ms)",
                              continuous_update=False),
             schedule=Dropdown(options=["(b) next tick", "(c) compute early"],
                               value="(c) compute early", description="schedule"))
except ImportError:
    h1_averaging()
    h2_roundoff()
    h3_fractional_delay()
    h4_phase_budget()